## Example 1: Version Comparison Utilities

Demonstrate version comparison functionality from lightning_utilities.

In [ ]:
# Example 1: Import and module availability checking
from lightning_utilities.core.imports import module_available, package_available, ModuleAvailableCache
import sys

print("=== Module Availability Checking ===")

# Test module_available() function - unique to lightning-utilities
test_modules = [
    ('sys', True),
    ('os', True),
    ('json', True),
    ('numpy', 'maybe'),
    ('torch', 'maybe'),
    ('nonexistent_module_xyz', False)
]

print("\nTesting module_available():")
for module_name, expected in test_modules:
    is_available = module_available(module_name)
    status = "✓ Available" if is_available else "✗ Not available"
    print(f"  {module_name:25} -> {status}")
    if expected != 'maybe':
        assert is_available == expected, f"module_available({module_name!r}) = {is_available}, expected {expected}"

print("\n=== Package Availability Checking ===")

# Test package_available() function - unique to lightning-utilities
test_packages = ['sys', 'os', 'pathlib', 'collections', 'fake_package']

print("\nTesting package_available():")
for package_name in test_packages:
    is_available = package_available(package_name)
    status = "✓ Available" if is_available else "✗ Not available"
    print(f"  {package_name:25} -> {status}")
    assert is_available == (package_name != 'fake_package'), f"package_available({package_name!r}) = {is_available}"

print("\n=== ModuleAvailableCache Usage ===")

# Test ModuleAvailableCache class - unique to lightning-utilities
# This caches availability checks for performance
# Note: Each cache instance is for a specific module
print("\nUsing ModuleAvailableCache for efficient repeated checks:")

modules_to_cache = ['os', 'sys', 'json', 'numpy', 'fake_module']
for module_name in modules_to_cache:
    # Create cache for specific module
    cache = ModuleAvailableCache(module_name)
    # Use as boolean to check availability (cached after first check)
    is_available = bool(cache)
    print(f"  {module_name}: {'Available' if is_available else 'Not available'}")
    if module_name != 'numpy':
        assert is_available == (module_name != 'fake_module'), f"ModuleAvailableCache({module_name!r}) = {is_available}"
    
    # Second check uses cached result
    is_available_cached = bool(cache)
    print(f"    (cached check: {is_available_cached})")
    assert is_available_cached == is_available, f"cached check for {module_name!r} disagrees with the first"
    
    # Can also get message as string
    message = str(cache)
    if message:
        print(f"    Message: {message}")

print("\n=== Conditional Imports Pattern ===")

# Demonstrate the unique conditional import pattern
print("\nConditional import pattern (lightning-utilities idiom):")

if module_available("os.path"):
    from os.path import exists, join
    print("  ✓ Imported os.path.exists and os.path.join")
    print(f"    Testing: exists('/') = {exists('/')}")
else:
    raise AssertionError("module_available('os.path') is False for a stdlib submodule")

if module_available("collections.abc"):
    from collections.abc import Mapping
    print("  ✓ Imported collections.abc.Mapping")
    print(f"    Testing: isinstance({{}}, Mapping) = {isinstance({}, Mapping)}")
else:
    raise AssertionError("module_available('collections.abc') is False for a stdlib submodule")

assert not module_available("nonexistent.fake.module"), "module_available reported a module that does not exist"
print("  ✓ Correctly detected nonexistent.fake.module is not available")

print("\n=== Benefits of lightning-utilities Approach ===")
print("1. Cleaner optional dependency handling")
print("2. Built-in caching for performance")
print("3. Consistent API across Lightning ecosystem")
print("4. Handles both packages and submodules")

print("\nExample 1: Import and module availability checking completed successfully")

## Example 2: Rank Zero Operations

Demonstrate rank-based operations for distributed training scenarios.

In [ ]:
# Example 2: Rank zero operations for distributed training
from lightning_utilities.core.rank_zero import (
    rank_zero_only,
    rank_zero_info,
    rank_zero_debug,
    rank_zero_warn,
    rank_zero_deprecation,
    WarningCache
)

print("=== Rank Zero Operations ===\n")

# IMPORTANT: Set the rank before using rank_zero_only decorator
# In distributed training, this would be set to the process rank (0, 1, 2, ...)
# For single-process execution, set it to 0
rank_zero_only.rank = 0

print("Setting rank to 0 (simulating main process)")
print(f"Current rank: {rank_zero_only.rank}\n")

# Test rank_zero_only decorator - unique to lightning-utilities
# This ensures functions only run on the main process in distributed training
@rank_zero_only
def train_step_logging():
    """Function that only executes on rank 0 to avoid duplicate logs"""
    return "Training step logged on rank 0"

@rank_zero_only
def save_checkpoint(path):
    """Save checkpoint only on rank 0 to avoid file conflicts"""
    return f"Checkpoint saved to {path} by rank 0"

print("Testing @rank_zero_only decorator:")
result1 = train_step_logging()
result2 = save_checkpoint("/tmp/model.ckpt")
print(f"  Training log: {result1}")
print(f"  Checkpoint save: {result2}")

print("\n=== Rank Zero Logging Functions ===")

# Test rank_zero_info - logs only from rank 0
print("\nrank_zero_info() - Info-level logging:")
rank_zero_info("Starting training epoch 1")
rank_zero_info("Batch size: 32, Learning rate: 0.001")

# Test rank_zero_debug - debug logging from rank 0 only
print("\nrank_zero_debug() - Debug-level logging:")
rank_zero_debug("Debug: Model weights initialized")
rank_zero_debug("Debug: Gradient accumulation steps = 4")

# Test rank_zero_warn - warnings from rank 0 only
print("\nrank_zero_warn() - Warning-level logging:")
rank_zero_warn("Warning: Learning rate is very high")
rank_zero_warn("Warning: Dataset might be too small")

# Test rank_zero_deprecation - deprecation warnings
print("\nrank_zero_deprecation() - Deprecation warnings:")
rank_zero_deprecation(
    "The old_api() function is deprecated. Use new_api() instead.",
    stacklevel=2
)

print("\n=== WarningCache for De-duplication ===")

# Test WarningCache - unique to lightning-utilities
# Prevents repeated warnings in distributed training
cache = WarningCache()

print("\nUsing WarningCache to avoid duplicate warnings:")
messages = [
    "CUDA not available, using CPU",
    "Dataset has missing values",
    "CUDA not available, using CPU",  # Duplicate
    "Model checkpoint not found",
    "Dataset has missing values",  # Duplicate
]

for msg in messages:
    if cache.warn(msg):
        print(f"  [NEW WARNING] {msg}")
    else:
        print(f"  [CACHED] {msg} (suppressed duplicate)")

print("\n=== Use Case: Distributed Training Simulation ===")

def simulate_distributed_training(rank):
    """Simulate a distributed training scenario"""
    print(f"\n--- Process Rank {rank} ---")
    
    # Set the rank for this process
    rank_zero_only.rank = rank
    
    # Only rank 0 should log these
    rank_zero_info(f"Epoch started (should only see once)")
    rank_zero_debug(f"Loading data (should only see once)")
    
    # All ranks do computation
    print(f"  Rank {rank}: Computing gradients...")
    
    # Only rank 0 saves
    result = save_checkpoint(f"/tmp/model_rank{rank}.ckpt")
    if result:
        print(f"  {result}")

# Simulate multiple processes
print("\nSimulating multi-process training:")
for process_rank in [0, 1, 2]:
    simulate_distributed_training(process_rank)

# Reset to rank 0 for remaining examples
rank_zero_only.rank = 0

print("\n=== Benefits for Distributed Training ===")
print("1. Prevents duplicate logs in multi-GPU training")
print("2. Ensures single checkpoint save to avoid conflicts")
print("3. De-duplicates warnings across processes")
print("4. Clean, explicit rank 0 operations")
print("5. Standard pattern across Lightning ecosystem")

print("\nExample 2: Rank zero operations completed successfully")

## Example 3: Import Utilities and Module Checking

Demonstrate import utilities for checking module availability.

In [ ]:
# Example 3: Version comparison and lazy imports
from lightning_utilities.core.imports import compare_version, lazy_import, LazyModule
import sys

print("=== Version Comparison Utilities ===")

# Test compare_version() function - unique to lightning-utilities
# This is commonly used to check package compatibility
print("\nTesting compare_version():")

# Compare Python version
python_version = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
print(f"Current Python version: {python_version}")

# Test version comparisons with operator
version_checks = [
    (">=", "3.8.0", "Python 3.8+ check"),
    (">=", "3.7.0", "Python 3.7+ check"),
    ("<", "4.0.0", "Python < 4.0 check"),
    ("==", python_version, f"Exact version check"),
]

for op, target_version, description in version_checks:
    # compare_version compares against actual installed package
    # We'll demonstrate the API pattern
    print(f"  {description}:")
    print(f"    Comparing {python_version} {op} {target_version}")
    
    # Simulate version comparison logic
    from packaging import version as pkg_version
    current = pkg_version.parse(python_version)
    target = pkg_version.parse(target_version)
    
    if op == ">=":
        result = current >= target
    elif op == "<=":
        result = current <= target
    elif op == ">":
        result = current > target
    elif op == "<":
        result = current < target
    elif op == "==":
        result = current == target
    else:
        result = False
    
    print(f"    Result: {result}")

print("\n=== Lazy Import Functionality ===")

# Test lazy_import() - unique to lightning-utilities
# Defers actual import until attribute access
print("\nTesting lazy_import():")

try:
    # Lazy import - module only loaded when accessed
    lazy_os = lazy_import("os")
    print(f"  Created lazy import for 'os' module")
    print(f"  Type: {type(lazy_os)}")
    
    # Access an attribute - this triggers actual import
    print(f"  Accessing os.name: {lazy_os.name}")
    print(f"  Accessing os.sep: {lazy_os.sep}")
    
    # Lazy import of submodule
    lazy_os_path = lazy_import("os.path")
    print(f"  Created lazy import for 'os.path'")
    print(f"  Accessing os.path.sep: {lazy_os_path.sep}")
    
except Exception as e:
    print(f"  Error: {e}")
    raise

# Test LazyModule class
print("\n=== LazyModule Class ===")

try:
    # Create LazyModule instance
    lazy_json = LazyModule("json")
    print(f"  Created LazyModule for 'json'")
    print(f"  Type: {type(lazy_json)}")
    
    # Module is imported on first attribute access
    test_data = {"key": "value", "number": 42}
    json_str = lazy_json.dumps(test_data)
    print(f"  Called lazy_json.dumps(): {json_str}")
    
    parsed = lazy_json.loads(json_str)
    print(f"  Called lazy_json.loads(): {parsed}")
    
except Exception as e:
    print(f"  Error: {e}")
    raise

print("\n=== Use Case: Conditional Feature Loading ===")

# Practical example: load optional dependencies only when needed
def process_with_numpy_if_available():
    """Process data with numpy if available, fallback otherwise"""
    from lightning_utilities.core.imports import module_available
    
    if module_available("numpy"):
        # Only import numpy if available
        np = lazy_import("numpy")
        data = [1, 2, 3, 4, 5]
        arr = np.array(data)
        print(f"  NumPy processing: mean = {np.mean(arr)}")
        return "numpy"
    else:
        # Fallback to standard library
        data = [1, 2, 3, 4, 5]
        mean = sum(data) / len(data)
        print(f"  Standard library processing: mean = {mean}")
        return "builtin"

print("\nConditional feature loading:")
backend = process_with_numpy_if_available()
print(f"  Backend used: {backend}")

print("\n=== Version Checking Pattern ===")

# Common pattern: check if package meets minimum version
def check_package_compatibility(package_name, min_version):
    """Check if package meets minimum version requirement"""
    from lightning_utilities.core.imports import module_available
    
    if not module_available(package_name):
        return False, "not installed"
    
    # In real usage, compare_version would check against installed version
    return True, "available"

print("\nChecking package compatibility:")
packages_to_check = [
    ("sys", "3.8.0"),
    ("os", "1.0.0"),
    ("json", "2.0.0"),
]

for pkg, min_ver in packages_to_check:
    available, status = check_package_compatibility(pkg, min_ver)
    symbol = "✓" if available else "✗"
    print(f"  {symbol} {pkg} >= {min_ver}: {status}")

print("\n=== Benefits of Lazy Imports ===")
print("1. Faster startup time - modules loaded only when needed")
print("2. Reduced memory footprint for optional features")
print("3. Graceful handling of missing dependencies")
print("4. Clean separation of optional and required imports")
print("5. Better compatibility across different environments")

print("\nExample 3: Version comparison and lazy imports completed successfully")

## Example 4: Package and Environment Warnings

Demonstrate warning utilities for package compatibility and environment issues.

In [ ]:
# Example 4: Apply functions to collections
from lightning_utilities.core.apply_func import (
    apply_to_collection,
    apply_to_collections,
    is_dataclass_instance,
    is_namedtuple
)
from collections import namedtuple
from dataclasses import dataclass

print("=== Apply to Collection Utilities ===")

# Test apply_to_collection() - unique to lightning-utilities
# Recursively applies a function to all elements of a specific type
print("\nTesting apply_to_collection():")

# Example 1: Apply function to all integers in nested structure
nested_data = {
    'a': [1, 2, 3],
    'b': {'x': 4, 'y': 5},
    'c': (6, 7, 8),
    'd': 'not an int'
}

print(f"  Original data: {nested_data}")

# Apply doubling to all integers
doubled = apply_to_collection(nested_data, int, lambda x: x * 2)
print(f"  After doubling all ints: {doubled}")

# Example 2: Apply to tuples specifically
# Using tuples instead of nested lists to avoid recursive application issues
nested_tuples = {
    'data': [(1, 2), (3, 4)],
    'more': [(5, 6)],
    'single': 42
}

print(f"\n  Original nested tuples: {nested_tuples}")

# Sum each tuple
summed = apply_to_collection(nested_tuples, tuple, lambda x: sum(x))
print(f"  After summing all tuples: {summed}")

# Example 3: Convert all strings to uppercase
string_data = {
    'name': 'alice',
    'items': ['apple', 'banana'],
    'nested': {'fruit': 'orange', 'count': 5}
}

print(f"\n  Original strings: {string_data}")
uppercased = apply_to_collection(string_data, str, lambda s: s.upper())
print(f"  After uppercasing: {uppercased}")

print("\n=== Apply to Collections (Pairwise) ===")

# Test apply_to_collections() - applies function to matching elements from two collections
print("\nTesting apply_to_collections():")

# Example: Element-wise addition of two nested structures
# IMPORTANT: Both structures must have the same shape for matching
data1 = {
    'a': [1, 2, 3],
    'b': {'x': 10, 'y': 20},
    'c': 100
}

data2 = {
    'a': [10, 20, 30],
    'b': {'x': 5, 'y': 15},
    'c': 50
}

print(f"  Data 1: {data1}")
print(f"  Data 2: {data2}")

# Add corresponding integers
result = apply_to_collections(data1, data2, int, lambda x, y: x + y)
print(f"  Element-wise sum: {result}")

# Example 2: Multiply corresponding floats
float_data1 = {
    'weights': [1.0, 2.0, 3.0],
    'bias': 0.5
}

float_data2 = {
    'weights': [2.0, 3.0, 4.0],
    'bias': 1.5
}

print(f"\n  Float data 1: {float_data1}")
print(f"  Float data 2: {float_data2}")

multiplied = apply_to_collections(float_data1, float_data2, float, lambda x, y: x * y)
print(f"  Element-wise multiplication: {multiplied}")

print("\n=== Type Checking Utilities ===")

# Test is_namedtuple() - unique to lightning-utilities
Person = namedtuple('Person', ['name', 'age'])
person = Person('Alice', 30)
regular_tuple = ('Alice', 30)

print("\nTesting is_namedtuple():")
print(f"  Person namedtuple: {is_namedtuple(person)}")
print(f"  Regular tuple: {is_namedtuple(regular_tuple)}")
print(f"  Dictionary: {is_namedtuple({'name': 'Alice'})}")

# Test is_dataclass_instance() - unique to lightning-utilities
@dataclass
class Config:
    learning_rate: float
    batch_size: int

config = Config(0.001, 32)
regular_class = type('RegularClass', (), {})()

print("\nTesting is_dataclass_instance():")
print(f"  Config dataclass: {is_dataclass_instance(config)}")
print(f"  Regular class: {is_dataclass_instance(regular_class)}")
print(f"  Dictionary: {is_dataclass_instance({'lr': 0.001})}")

print("\n=== Real-world Use Case: Processing Model Parameters ===")

# Simulate processing model parameters with nested structure
model_params = {
    'encoder': {
        'layer1': {'weights': [0.1, 0.2, 0.3], 'bias': [0.01]},
        'layer2': {'weights': [0.4, 0.5], 'bias': [0.02]}
    },
    'decoder': {
        'weights': [0.6, 0.7, 0.8, 0.9],
        'bias': [0.03]
    },
    'name': 'transformer'
}

print("\nOriginal model parameters:")
print(f"  {model_params}")

# Apply gradient clipping to all floats
def clip_gradient(x, min_val=-1.0, max_val=1.0):
    """Clip values to range"""
    return max(min_val, min(max_val, x))

clipped = apply_to_collection(model_params, float, clip_gradient)
print(f"\nAfter gradient clipping:")
print(f"  {clipped}")

# Count total number of parameters
def count_params(data):
    """Count all numeric parameters"""
    counter = {'count': 0}
    
    def increment(x):
        counter['count'] += 1
        return x
    
    apply_to_collection(data, float, increment)
    return counter['count']

param_count = count_params(model_params)
print(f"\nTotal float parameters: {param_count}")

print("\n=== Use Case: Gradient Accumulation ===")

# Simulate gradient accumulation from two batches
gradients_batch1 = {
    'layer1': [0.1, 0.2, 0.3],
    'layer2': [0.4, 0.5]
}

gradients_batch2 = {
    'layer1': [0.05, 0.15, 0.25],
    'layer2': [0.3, 0.35]
}

print("\nGradient accumulation example:")
print(f"  Batch 1 gradients: {gradients_batch1}")
print(f"  Batch 2 gradients: {gradients_batch2}")

# Accumulate gradients element-wise
accumulated = apply_to_collections(
    gradients_batch1,
    gradients_batch2,
    float,
    lambda g1, g2: g1 + g2
)

print(f"  Accumulated gradients: {accumulated}")

# Average the accumulated gradients
averaged = apply_to_collection(accumulated, float, lambda x: x / 2.0)
print(f"  Averaged gradients: {averaged}")

print("\n=== Benefits of apply_to_collection ===")
print("1. Process deeply nested data structures easily")
print("2. Type-safe transformations")
print("3. Preserves structure while transforming values")
print("4. Works with lists, dicts, tuples, namedtuples, dataclasses")
print("5. Essential for ML/DL parameter processing")

print("\nExample 4: Apply functions to collections completed successfully")

## Example 5: Enums and Utility Classes

Demonstrate enum definitions and utility classes from lightning_utilities.

In [ ]:
# Example 5: StrEnum and requirement utilities
from lightning_utilities.core.enums import StrEnum
from lightning_utilities.core.imports import RequirementCache, requires

print("=== StrEnum - String-based Enumerations ===")

# Test StrEnum - unique to lightning-utilities
# Provides flexible string-based enums with case-insensitive comparison
print("\nDefining enums with StrEnum:")

class ModelType(StrEnum):
    """Model architecture types"""
    TRANSFORMER = "transformer"
    CNN = "cnn"
    RNN = "rnn"
    LSTM = "lstm"

class DatasetType(StrEnum):
    """Dataset categories"""
    TRAIN = "train"
    VAL = "validation"
    TEST = "test"

# Display enum values
print("  ModelType enum values:")
for model in ModelType:
    print(f"    {model.name} = {model.value}")

print("\n  DatasetType enum values:")
for dataset in DatasetType:
    print(f"    {dataset.name} = {dataset.value}")

print("\n=== StrEnum.from_str() - Flexible String Conversion ===")

# Test from_str() method - converts strings to enum
print("\nConverting strings to enums:")

test_strings = [
    ("transformer", "exact match"),
    ("TRANSFORMER", "uppercase"),
    ("Transformer", "mixed case"),
    ("CNN", "by name"),
    ("cnn", "by value")
]

for test_str, description in test_strings:
    try:
        model = ModelType.from_str(test_str)
        print(f"  '{test_str}' ({description}) -> {model}")
    except ValueError as e:
        print(f"  '{test_str}' ({description}) -> Error: {e}")

print("\n=== StrEnum.try_from_str() - Safe Conversion ===")

# Test try_from_str() - returns None instead of raising error
print("\nSafe string to enum conversion:")

safe_test_strings = ['lstm', 'INVALID_MODEL', 'rnn', 'GPT']

for test_str in safe_test_strings:
    result = ModelType.try_from_str(test_str)
    if result:
        print(f"  '{test_str}' -> {result} ✓")
    else:
        print(f"  '{test_str}' -> None (not found)")

print("\n=== StrEnum Equality Comparison ===")

# Test flexible equality comparison
print("\nTesting equality comparisons:")

transformer_enum = ModelType.TRANSFORMER

comparisons = [
    (transformer_enum == "transformer", "enum == lowercase string"),
    (transformer_enum == "TRANSFORMER", "enum == uppercase string"),
    (transformer_enum == "Transformer", "enum == mixed case"),
    (transformer_enum == ModelType.TRANSFORMER, "enum == same enum"),
    (transformer_enum == ModelType.CNN, "enum == different enum"),
    (transformer_enum == "rnn", "enum == different string")
]

for result, description in comparisons:
    symbol = "✓" if result else "✗"
    print(f"  {symbol} {description}: {result}")

print("\n=== RequirementCache - Dependency Checking ===")

# Test RequirementCache - caches package requirement checks
# Each instance checks a specific requirement
print("\nTesting RequirementCache:")

# Check various package requirements
requirements = [
    "sys",
    "os",
    "json",
    "numpy>=1.0",
    "nonexistent_pkg>=1.0"
]

print("  Checking requirements:")
for req in requirements:
    # Create cache for specific requirement
    req_cache = RequirementCache(req)
    # Use as boolean to check if satisfied
    is_satisfied = bool(req_cache)
    symbol = "✓" if is_satisfied else "✗"
    print(f"    {symbol} {req}: {'satisfied' if is_satisfied else 'not satisfied'}")
    
    # Can also get detailed message
    if not is_satisfied:
        message = str(req_cache)
        print(f"       Message: {message}")

print("\n=== RequirementCache with Module Checking ===")

# Test module-based checking
print("\nChecking module availability:")

modules_to_check = [
    "os.path",
    "json",
    "sys",
    "fake.module"
]

for module_name in modules_to_check:
    # Create cache for specific module
    mod_cache = RequirementCache(module=module_name)
    is_available = bool(mod_cache)
    symbol = "✓" if is_available else "✗"
    print(f"  {symbol} Module '{module_name}': {'available' if is_available else 'not available'}")

print("\n=== @requires Decorator ===")

# Test @requires decorator - marks function as requiring specific packages
print("\nTesting @requires decorator:")

try:
    @requires("os")
    def use_os_features():
        """Function that requires os package"""
        import os
        return f"OS name: {os.name}"

    @requires("fake_package>=2.0")
    def use_fake_package():
        """Function that requires nonexistent package"""
        return "This won't run"

    print("  Calling function with satisfied requirement:")
    result1 = use_os_features()
    print(f"    {result1}")

    print("  Calling function with unsatisfied requirement:")
    try:
        result2 = use_fake_package()
    except ModuleNotFoundError as e:
        assert "fake_package>=2.0" in str(e), e
        print(f"    Caught expected error: {type(e).__name__}")
    else:
        raise AssertionError(f"@requires let a function with an unsatisfied requirement run: {result2!r}")

except Exception as e:
    print(f"  Note: @requires decorator may need specific setup: {type(e).__name__}")
    # Re-raise: this block guards nothing optional (`os` is stdlib), and swallowing here would also
    # swallow the AssertionError above that proves `@requires` refuses an unsatisfied requirement.
    raise

print("\n=== Practical Use Case: Configuration Management ===")

# Use StrEnum for configuration values
class Environment(StrEnum):
    """Deployment environments"""
    DEVELOPMENT = "dev"
    STAGING = "staging"
    PRODUCTION = "prod"

class LogLevel(StrEnum):
    """Logging levels"""
    DEBUG = "debug"
    INFO = "info"
    WARNING = "warning"
    ERROR = "error"

# Simulate configuration from string inputs
config_strings = {
    'environment': 'PRODUCTION',
    'log_level': 'info'
}

print("\nParsing configuration:")
print(f"  Raw config: {config_strings}")

parsed_config = {
    'environment': Environment.from_str(config_strings['environment']),
    'log_level': LogLevel.from_str(config_strings['log_level'])
}

print(f"  Parsed config:")
print(f"    Environment: {parsed_config['environment']}")
print(f"    Log level: {parsed_config['log_level']}")

# Use in conditional logic
if parsed_config['environment'] == Environment.PRODUCTION:
    print("  ✓ Running in production mode")
if parsed_config['log_level'] == "info":
    print("  ✓ Info logging enabled")

print("\n=== Use Case: Model Selection ===")

def create_model(model_type_str: str):
    """Factory function using StrEnum for model selection"""
    model_type = ModelType.try_from_str(model_type_str)
    
    if model_type == ModelType.TRANSFORMER:
        return "TransformerModel()"
    elif model_type == ModelType.CNN:
        return "CNNModel()"
    elif model_type == ModelType.RNN:
        return "RNNModel()"
    elif model_type == ModelType.LSTM:
        return "LSTMModel()"
    else:
        return "DefaultModel()"

print("\nModel factory with StrEnum:")
for model_str in ['transformer', 'CNN', 'lstm', 'invalid']:
    model = create_model(model_str)
    print(f"  create_model('{model_str}') -> {model}")

print("\n=== Use Case: Conditional Feature Imports ===")

# Use RequirementCache to conditionally enable features
print("\nConditional feature loading based on requirements:")

# Check if numpy is available
numpy_req = RequirementCache("numpy")
if bool(numpy_req):
    print("  ✓ NumPy available - enabling advanced numerical features")
else:
    print(f"  ✗ NumPy not available - using basic features only")

# Check if torch is available with version requirement
torch_req = RequirementCache("torch>=1.0")
if bool(torch_req):
    print("  ✓ PyTorch 1.0+ available - enabling deep learning features")
else:
    print(f"  ✗ PyTorch 1.0+ not available")

print("\n=== Benefits of StrEnum ===")
print("1. Case-insensitive string comparison")
print("2. Flexible string-to-enum conversion")
print("3. Safe conversion with try_from_str()")
print("4. Natural equality checks with strings")
print("5. Perfect for config parsing and CLI args")

print("\n=== Benefits of RequirementCache ===")
print("1. Efficient caching of dependency checks")
print("2. Version-aware requirement checking")
print("3. Decorator support with @requires")
print("4. Consistent with Lightning ecosystem patterns")
print("5. Works with both packages and modules")

print("\nExample 5: StrEnum and requirement utilities completed successfully")